# Advanced Problems with Solutions: Single Dispatch Generic Functions

This notebook contains advanced practice problems on single dispatch, generic functions, dispatch registries, recursive rendering, type-based specialization, abstract base classes, and `functools.singledispatch`.

Each problem includes a full solution and runnable examples.

## Problem 1 — Manual HTML Dispatcher

Write a function called `htmlize(arg)` that dispatches manually based on the runtime type of `arg`.

Requirements:

- `int` values should render as decimal plus hexadecimal.
- `float` values should render rounded to two decimal places.
- `str` values should be HTML-escaped and newline characters should become `<br/>`.
- `list` and `tuple` values should render as unordered HTML lists.
- `dict` values should render key-value pairs as unordered HTML lists.
- All other objects should fall back to their escaped string representation.

Best-practice requirement: nested containers should be rendered recursively using `htmlize`.

In [1]:
from html import escape


def html_escape(arg):
    return escape(str(arg))


def html_int(value):
    return f'{value}(<i>{hex(value)}</i>)'


def html_float(value):
    return f'{value:.2f}'


def html_str(value):
    return html_escape(value).replace('\n', '<br/>\n')


def html_sequence(seq):
    items = [f'<li>{htmlize(item)}</li>' for item in seq]
    return '<ul>\n' + '\n'.join(items) + '\n</ul>'


def html_dict(d):
    items = [
        f'<li>{htmlize(key)}={htmlize(value)}</li>'
        for key, value in d.items()
    ]
    return '<ul>\n' + '\n'.join(items) + '\n</ul>'


def htmlize(arg):
    if isinstance(arg, int):
        return html_int(arg)
    elif isinstance(arg, float):
        return html_float(arg)
    elif isinstance(arg, str):
        return html_str(arg)
    elif isinstance(arg, (list, tuple)):
        return html_sequence(arg)
    elif isinstance(arg, dict):
        return html_dict(arg)
    else:
        return html_escape(arg)


print(htmlize(['a < b', 100, 3.14159, ('x', 255)]))

<ul>
<li>a &lt; b</li>
<li>100(<i>0x64</i>)</li>
<li>3.14</li>
<li><ul>
<li>x</li>
<li>255(<i>0xff</i>)</li>
</ul></li>
</ul>


### Solution Explanation

The function `htmlize` is a dispatcher: it chooses a type-specific implementation based on the type of its argument.

The important detail is recursion. The sequence and dictionary renderers call `htmlize` again for their elements, so nested values are handled by the most appropriate renderer.

## Problem 2 — Build a Minimal Single Dispatch Decorator

Implement a simplified `single_dispatch` decorator.

It should:

- Store the original decorated function as the default handler for `object`.
- Return a wrapper that dispatches based on `type(arg)`.
- Expose a `.register(type_)` decorator used to register specialized handlers.
- Expose a `.dispatch(type_)` function that returns the registered function for a type, or the default.
- Expose a `.registry` dictionary for inspection.

This first version only needs exact type matching.

In [2]:
from functools import wraps


def single_dispatch(fn):
    registry = {object: fn}

    @wraps(fn)
    def wrapper(arg):
        handler = registry.get(type(arg), registry[object])
        return handler(arg)

    def register(type_):
        def decorator(handler):
            registry[type_] = handler
            return handler
        return decorator

    def dispatch(type_):
        return registry.get(type_, registry[object])

    wrapper.register = register
    wrapper.dispatch = dispatch
    wrapper.registry = registry
    return wrapper


@single_dispatch
def render(arg):
    return escape(str(arg))


@render.register(int)
def render_int(arg):
    return f'{arg}({hex(arg)})'


@render.register(str)
def render_str(arg):
    return escape(arg).replace('\n', '<br/>\n')


render(100), render('a < b'), render(3 + 4j), render.registry

('100(0x64)',
 'a &lt; b',
 '(3+4j)',
 {object: <function __main__.render(arg)>,
  int: <function __main__.render_int(arg)>,
  str: <function __main__.render_str(arg)>})

### Solution Explanation

The registry is stored in the closure of the decorated function. The returned wrapper has extra attributes attached to it: `.register`, `.dispatch`, and `.registry`.

This mirrors the public style of `functools.singledispatch`, although this version is intentionally simpler.

## Problem 3 — Why the Register Decorator Must Return the Handler

Modify the previous `single_dispatch` implementation so that register decorators can be stacked.

Then register the same function for both `list` and `tuple`.

Best-practice requirement: the inner registration decorator must return the original handler function.

In [3]:
@render.register(tuple)
@render.register(list)
def render_sequence(seq):
    items = [f'<li>{render(item)}</li>' for item in seq]
    return '<ul>\n' + '\n'.join(items) + '\n</ul>'


print(render([1, 'a < b', 3 + 4j]))
print(render((1, 2, 3)))

<ul>
<li>1(0x1)</li>
<li>a &lt; b</li>
<li>(3+4j)</li>
</ul>
<ul>
<li>1(0x1)</li>
<li>2(0x2)</li>
<li>3(0x3)</li>
</ul>


### Solution Explanation

Stacked decorators are applied from bottom to top.

This code:

```python
@render.register(tuple)
@render.register(list)
def render_sequence(seq):
    ...
```

is equivalent to:

```python
render_sequence = render.register(tuple)(render.register(list)(render_sequence))
```

Therefore, `render.register(list)(render_sequence)` must return `render_sequence`, otherwise the next decorator would receive `None` instead of the function.

## Problem 4 — Support Multiple Positional and Keyword Arguments

Improve the custom `single_dispatch` decorator so that it can wrap functions accepting multiple positional and keyword arguments.

Dispatch should still be based only on the type of the first positional argument.

Requirements:

- Support `*args` and `**kwargs`.
- Raise a clear `TypeError` if no positional argument is provided.
- Preserve function metadata with `functools.wraps`.

In [4]:
def single_dispatch_multi(fn):
    registry = {object: fn}

    @wraps(fn)
    def wrapper(*args, **kwargs):
        if not args:
            raise TypeError('single dispatch requires at least one positional argument')
        first_arg = args[0]
        handler = registry.get(type(first_arg), registry[object])
        return handler(*args, **kwargs)

    def register(type_):
        def decorator(handler):
            registry[type_] = handler
            return handler
        return decorator

    def dispatch(type_):
        return registry.get(type_, registry[object])

    wrapper.register = register
    wrapper.dispatch = dispatch
    wrapper.registry = registry
    return wrapper


@single_dispatch_multi
def describe(value, *, label='value'):
    return f'{label}: {value!r}'


@describe.register(int)
def _(value, *, label='integer'):
    return f'{label}: decimal={value}, hex={hex(value)}'


describe(255), describe(255, label='number'), describe(object(), label='object')

('integer: decimal=255, hex=0xff',
 'number: decimal=255, hex=0xff',
 'object: <object object at 0x000001D604ED5490>')

In [5]:
try:
    describe(label='missing positional argument')
except TypeError as ex:
    print(type(ex).__name__, ex)

TypeError single dispatch requires at least one positional argument


### Solution Explanation

Single dispatch is based on one selected argument. In Python's `functools.singledispatch`, that argument is the first positional argument.

This implementation follows the same idea: it dispatches on `args[0]`, then passes the entire argument list to the selected handler.

## Problem 5 — Add Inheritance-Aware Dispatch

The previous custom dispatcher only checks `type(arg)` exactly.

Improve it so that a registered parent class can handle subclass instances.

Use the argument type's Method Resolution Order, available through `type(arg).__mro__`.

Requirements:

- Prefer the most specific registered type.
- Fall back to `object`.
- Support custom subclasses.

In [6]:
def single_dispatch_mro(fn):
    registry = {object: fn}

    @wraps(fn)
    def wrapper(*args, **kwargs):
        if not args:
            raise TypeError('single dispatch requires at least one positional argument')
        handler = dispatch(type(args[0]))
        return handler(*args, **kwargs)

    def register(type_):
        def decorator(handler):
            registry[type_] = handler
            return handler
        return decorator

    def dispatch(type_):
        for cls in type_.__mro__:
            if cls in registry:
                return registry[cls]
        return registry[object]

    wrapper.register = register
    wrapper.dispatch = dispatch
    wrapper.registry = registry
    return wrapper


class Animal:
    pass


class Dog(Animal):
    pass


class Cat(Animal):
    pass


@single_dispatch_mro
def speak(animal):
    return 'unknown object'


@speak.register(Animal)
def _(animal):
    return 'generic animal sound'


@speak.register(Dog)
def _(dog):
    return 'woof'


speak(Dog()), speak(Cat()), speak(object())

('woof', 'generic animal sound', 'unknown object')

### Solution Explanation

The MRO lists classes from most specific to least specific.

For a `Dog`, the MRO includes `Dog`, then `Animal`, then `object`. Therefore, if `Dog` is registered, it wins. If not, `Animal` can still handle the object.

## Problem 6 — Use `functools.singledispatch` with Abstract Base Classes

Use Python's built-in `functools.singledispatch` to create a robust `htmlize` function.

Requirements:

- Use `numbers.Integral` for integer-like values.
- Use `numbers.Real` for real numbers.
- Use `collections.abc.Sequence` for list-like values.
- Add a specific handler for `str` to avoid infinite recursion.
- Add a specific handler for `tuple` that renders tuples differently from lists.
- Add a handler for `dict`.

Best-practice warning: because `str` is also a sequence, always register a specific `str` handler when registering a generic sequence handler.

In [7]:
from functools import singledispatch
from numbers import Integral, Real
from collections.abc import Sequence


@singledispatch
def htmlize(value):
    return escape(str(value))


@htmlize.register(Integral)
def _(value):
    return f'{value}(<i>{hex(value)}</i>)'


@htmlize.register(Real)
def _(value):
    return f'{value:.2f}'


@htmlize.register(str)
def _(value):
    return escape(value).replace('\n', '<br/>\n')


@htmlize.register(Sequence)
def _(seq):
    items = [f'<li>{htmlize(item)}</li>' for item in seq]
    return '<ul>\n' + '\n'.join(items) + '\n</ul>'


@htmlize.register(tuple)
def _(seq):
    items = ', '.join(htmlize(item) for item in seq)
    return f'({items})'


@htmlize.register(dict)
def _(d):
    items = [
        f'<li>{htmlize(key)}={htmlize(value)}</li>'
        for key, value in d.items()
    ]
    return '<ul>\n' + '\n'.join(items) + '\n</ul>'


print(htmlize(['a < b', 255, 3.14159, ('x', 10)]))

<ul>
<li>a &lt; b</li>
<li>255(<i>0xff</i>)</li>
<li>3.14</li>
<li>(x, 10(<i>0xa</i>))</li>
</ul>


In [8]:
htmlize.dispatch(int), htmlize.dispatch(bool), htmlize.dispatch(list), htmlize.dispatch(str), htmlize.dispatch(tuple)

(<function __main__._(value)>,
 <function __main__._(value)>,
 <function __main__._(seq)>,
 <function __main__._(value)>,
 <function __main__._(seq)>)

### Solution Explanation

`functools.singledispatch` supports inheritance and abstract base classes. Registering `Integral` handles both `int` and `bool`, because `bool` is a subclass of `int` and is considered integral.

The `str` handler is necessary because strings are sequences. Without the specific `str` registration, the sequence handler would iterate over characters and recursively dispatch each character forever.

## Problem 7 — Avoid the `bool` Is an `int` Trap

In Python, `bool` is a subclass of `int`. When using `Integral`, `True` and `False` will be handled by the integer renderer.

Create a `htmlize` function where:

- `bool` renders as `<b>True</b>` or `<b>False</b>`.
- other integral values render as decimal plus hexadecimal.

Use `functools.singledispatch`.

In [9]:
@singledispatch
def htmlize_bool_safe(value):
    return escape(str(value))


@htmlize_bool_safe.register(Integral)
def _(value):
    return f'{value}(<i>{hex(value)}</i>)'


@htmlize_bool_safe.register(bool)
def _(value):
    return f'<b>{value}</b>'


htmlize_bool_safe(True), htmlize_bool_safe(False), htmlize_bool_safe(10), htmlize_bool_safe.dispatch(bool), htmlize_bool_safe.dispatch(int)

('<b>True</b>',
 '<b>False</b>',
 '10(<i>0xa</i>)',
 <function __main__._(value)>,
 <function __main__._(value)>)

### Solution Explanation

`singledispatch` chooses the most specific applicable registered type.

Since `bool` is more specific than `Integral`, registering `bool` separately overrides the more general integral handler for Boolean values.

## Problem 8 — Register Handlers with Type Annotations

`functools.singledispatch` can infer the registered type from a function annotation.

Create a generic function called `summarize(value)`.

Register handlers using annotations for:

- `str`
- `list`
- `dict`

Each handler should return a short summary string.

In [10]:
@singledispatch
def summarize(value):
    return f'object of type {type(value).__name__}'


@summarize.register
def _(value: str):
    return f'string with {len(value)} characters'


@summarize.register
def _(value: list):
    return f'list with {len(value)} items'


@summarize.register
def _(value: dict):
    return f'dict with {len(value)} keys'


summarize('hello'), summarize([1, 2, 3]), summarize({'a': 1}), summarize(3.14)

('string with 5 characters',
 'list with 3 items',
 'dict with 1 keys',
 'object of type float')

### Solution Explanation

When the specialized function has an annotation on its first argument, `@generic.register` can infer the type automatically.

This is often cleaner than writing `@generic.register(str)`, especially when the function signature is already annotated.

## Problem 9 — Debug the Registry and Dispatch Resolution

Create a helper function called `show_dispatch_table(generic_fn, types)`.

It should print, for each type in `types`, which function the generic function will dispatch to.

Then use it to inspect dispatch for:

- `object`
- `int`
- `bool`
- `str`
- `list`
- `tuple`
- `complex`

In [11]:
def show_dispatch_table(generic_fn, types):
    for type_ in types:
        handler = generic_fn.dispatch(type_)
        print(f'{type_.__name__:>10} -> {handler.__name__}')


show_dispatch_table(
    htmlize,
    [object, int, bool, str, list, tuple, complex]
)

    object -> htmlize
       int -> _
      bool -> _
       str -> _
      list -> _
     tuple -> _
   complex -> htmlize


### Solution Explanation

The `.dispatch(type_)` method is useful for debugging. It tells you which function will handle objects of that type.

The `.registry` attribute is useful for seeing what was explicitly registered, while `.dispatch` shows the effective resolution after considering inheritance and abstract base classes.

## Problem 10 — Use Single Dispatch for a Mini Serializer

Build a generic function called `to_data(value)` that converts Python objects into JSON-compatible data.

Requirements:

- `None`, `str`, `int`, `float`, and `bool` should return unchanged.
- `list` and `tuple` should recursively convert each item.
- `dict` should recursively convert values and convert keys to strings.
- `set` should convert to a sorted list when possible.
- `datetime` should convert to ISO 8601 strings.
- Unknown objects with a `__dict__` should convert their instance attributes recursively.
- Unknown objects without `__dict__` should convert to strings.

In [12]:
from datetime import datetime, timezone


@singledispatch
def to_data(value):
    if hasattr(value, '__dict__'):
        return {
            key: to_data(val)
            for key, val in vars(value).items()
        }
    return str(value)


@to_data.register(type(None))
@to_data.register(str)
@to_data.register(int)
@to_data.register(float)
@to_data.register(bool)
def _(value):
    return value


@to_data.register(list)
@to_data.register(tuple)
def _(value):
    return [to_data(item) for item in value]


@to_data.register(dict)
def _(value):
    return {
        str(key): to_data(val)
        for key, val in value.items()
    }


@to_data.register(set)
def _(value):
    converted = [to_data(item) for item in value]
    try:
        return sorted(converted)
    except TypeError:
        return converted


@to_data.register(datetime)
def _(value):
    return value.isoformat()


class User:
    def __init__(self, name, created_at, tags):
        self.name = name
        self.created_at = created_at
        self.tags = tags


user = User('Ada', datetime(2026, 1, 1, tzinfo=timezone.utc), {'python', 'math'})

to_data({
    'user': user,
    'scores': (10, 20, 30),
    'active': True,
    'metadata': None
})

{'user': {'name': 'Ada',
  'created_at': '2026-01-01T00:00:00+00:00',
  'tags': ['math', 'python']},
 'scores': [10, 20, 30],
 'active': True,
 'metadata': None}

### Solution Explanation

This is a practical use case for single dispatch: each type has a different conversion rule, but the public API remains one function: `to_data`.

Recursive calls allow nested structures to be converted deeply.

The generic fallback handles unknown objects by checking for `__dict__`, which works for many ordinary Python objects.

## Problem 11 — Create a Custom Pretty Printer with Registration

Create a `pretty(value)` generic function.

Requirements:

- Default: return `repr(value)`.
- `str`: wrap the escaped string in double quotes.
- `list`: pretty-print each item recursively.
- `dict`: pretty-print key-value pairs recursively.
- A custom `Point` class should render as `Point[x=..., y=...]`.

Use `functools.singledispatch`.

In [13]:
@singledispatch
def pretty(value):
    return repr(value)


@pretty.register(str)
def _(value):
    return f'"{escape(value)}"'


@pretty.register(list)
def _(value):
    return '[' + ', '.join(pretty(item) for item in value) + ']'


@pretty.register(dict)
def _(value):
    parts = [
        f'{pretty(key)}: {pretty(val)}'
        for key, val in value.items()
    ]
    return '{' + ', '.join(parts) + '}'


class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y


@pretty.register(Point)
def _(value):
    return f'Point[x={value.x}, y={value.y}]'


pretty({'name': 'Ada <admin>', 'points': [Point(1, 2), Point(3, 4)]})

'{"name": "Ada &lt;admin&gt;", "points": [Point[x=1, y=2], Point[x=3, y=4]]}'

### Solution Explanation

Single dispatch lets users add formatting for custom classes without changing the original `pretty` function.

This is the Open/Closed Principle in practice: the generic function is open for extension but closed for modification.

## Problem 12 — Explain Why This Causes Infinite Recursion

Consider this generic function:

```python
@singledispatch
def bad_htmlize(value):
    return escape(str(value))

@bad_htmlize.register(Sequence)
def _(seq):
    return ''.join(bad_htmlize(item) for item in seq)
```

Why does calling `bad_htmlize('abc')` cause infinite recursion?

Fix the implementation.

In [14]:
@singledispatch
def fixed_htmlize(value):
    return escape(str(value))


@fixed_htmlize.register(Sequence)
def _(seq):
    items = [f'<li>{fixed_htmlize(item)}</li>' for item in seq]
    return '<ul>\n' + '\n'.join(items) + '\n</ul>'


@fixed_htmlize.register(str)
def _(value):
    return escape(value).replace('\n', '<br/>\n')


fixed_htmlize('abc'), fixed_htmlize(['abc', 'x < y'])

('abc', '<ul>\n<li>abc</li>\n<li>x &lt; y</li>\n</ul>')

### Solution Explanation

`str` is a `Sequence`. If only a `Sequence` handler is registered, then a string is treated as a sequence of characters.

For `'abc'`, the sequence handler calls `bad_htmlize('a')`. But `'a'` is also a string, hence also a sequence, so it again calls the sequence handler. This continues until Python raises `RecursionError`.

The fix is to register a more specific `str` handler. `singledispatch` chooses the most specific applicable handler, so `str` will beat `Sequence` for strings.

# Summary

Single dispatch allows one public function name to support many type-specific implementations.

Best practices:

- Keep the generic default safe and simple.
- Register specialized functions instead of growing long `if...elif` chains.
- Use `functools.singledispatch` for production code.
- Use abstract base classes such as `Integral`, `Real`, and `Sequence` when appropriate.
- Register more specific handlers for special cases such as `bool`, `str`, or `tuple`.
- Use `.dispatch(type_)` and `.registry` to inspect behavior.
- Be careful with recursive handlers for containers.